# RET503 — Praktik Terpadu: Detektor YOLO (kamera vs mouse)
Alur: **Tahap 1–2** cek data → **Tahap 3–4** latih E1/E2/E3 → **Tahap 5** pilih model + uji test → **Tahap 6** laporan.

Sebelum mulai: **Runtime → Change runtime type → T4 GPU**, lalu jalankan sel dari atas ke bawah.

## 0. Persiapan

In [ ]:
!nvidia-smi -L
!pip install -q -U ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# ISI IDENTITAS KAMU
NIM  = "ISI_NIM"
NAMA = "ISI_NAMA"

## 1. Upload dataset
Upload `dataset_v1.zip` (isinya `images/`, `labels/`, `data.yaml`).

In [ ]:
from google.colab import files
up = files.upload()          # pilih dataset_v1.zip
!unzip -oq dataset_v1.zip -d /content
!ls /content/dataset_v1

## 2. Tahap 1 — Cek label (setara program 01)
Setiap gambar harus punya label, nomor kelas harus 0/1, dan nilai kotak harus 0–1.

In [ ]:
import glob, os, collections
R = "/content/dataset_v1"
DATA = f"{R}/data.yaml"
names = {0: "kamera", 1: "mouse"}

for s in ["train", "val", "test"]:
    imgs = sorted(glob.glob(f"{R}/images/{s}/*.jpg"))
    jml, masalah = collections.Counter(), []
    for im in imgs:
        lb = im.replace("/images/", "/labels/")[:-4] + ".txt"
        if not os.path.exists(lb):
            masalah.append(("label hilang", im)); continue
        for baris in open(lb).read().strip().splitlines():
            c, cx, cy, w, h = baris.split(); c = int(c); cx, cy, w, h = map(float, (cx, cy, w, h))
            if c not in names or not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
                masalah.append(("nilai salah", lb))
            jml[names[c]] += 1
    print(f"{s:5s} {len(imgs):3d} gambar  kotak: {dict(jml)}  masalah: {masalah or 'tidak ada'}")

In [ ]:
# Lihat beberapa gambar beserta kotaknya
import cv2, random, matplotlib.pyplot as plt
random.seed(0)
sampel = random.sample(sorted(glob.glob(f"{R}/images/train/*.jpg")), 8)
plt.figure(figsize=(16, 8))
for i, f in enumerate(sampel):
    im = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); H, W = im.shape[:2]
    c, cx, cy, w, h = open(f.replace("/images/", "/labels/")[:-4] + ".txt").read().split()
    cx, cy, w, h = float(cx)*W, float(cy)*H, float(w)*W, float(h)*H
    cv2.rectangle(im, (int(cx-w/2), int(cy-h/2)), (int(cx+w/2), int(cy+h/2)), (0, 255, 0), 12)
    plt.subplot(2, 4, i+1); plt.imshow(im); plt.title(names[int(c)]); plt.axis("off")
plt.show()

## 3. Tahap 1 — Konversi YOLO ↔ COCO (setara program 02 & 03)
Uji konversi dua arah: YOLO → COCO (piksel, kelas mulai 1) → YOLO lagi. Hasilnya harus sama dengan label asli.

In [ ]:
import json

def yolo_ke_coco(root, out_json):
    images, anns, aid, iid = [], [], 1, 1
    cats = [{"id": i+1, "name": n} for i, n in names.items()]      # COCO: kelas mulai 1
    for s in ["train", "val", "test"]:
        for im in sorted(glob.glob(f"{root}/images/{s}/*.jpg")):
            h, w = cv2.imread(im).shape[:2]
            images.append({"id": iid, "file_name": f"{s}/{os.path.basename(im)}", "width": w, "height": h})
            for baris in open(im.replace("/images/", "/labels/")[:-4] + ".txt").read().strip().splitlines():
                c, cx, cy, bw, bh = baris.split(); c = int(c); cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
                x, y, ww, hh = (cx-bw/2)*w, (cy-bh/2)*h, bw*w, bh*h  # pojok kiri atas, piksel
                anns.append({"id": aid, "image_id": iid, "category_id": c+1,
                             "bbox": [x, y, ww, hh], "area": ww*hh, "iscrowd": 0}); aid += 1
            iid += 1
    json.dump({"images": images, "annotations": anns, "categories": cats}, open(out_json, "w"))
    return len(images), len(anns)

def coco_ke_yolo(json_path, out_dir):
    d = json.load(open(json_path)); info = {i["id"]: i for i in d["images"]}
    for a in d["annotations"]:
        i = info[a["image_id"]]; x, y, w, h = a["bbox"]
        cx, cy, bw, bh = (x+w/2)/i["width"], (y+h/2)/i["height"], w/i["width"], h/i["height"]
        s, nama = i["file_name"].split("/")
        os.makedirs(f"{out_dir}/{s}", exist_ok=True)
        open(f"{out_dir}/{s}/{nama[:-4]}.txt", "a").write(f"{a['category_id']-1} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

!rm -rf /content/hasil_balik
print("gambar, kotak:", yolo_ke_coco(R, "/content/annotations_coco.json"))
coco_ke_yolo("/content/annotations_coco.json", "/content/hasil_balik")

selisih = 0
for f in glob.glob(f"{R}/labels/*/*.txt"):
    s = f.split("/")[-2]
    a = list(map(float, open(f).read().split()))
    b = list(map(float, open(f"/content/hasil_balik/{s}/{os.path.basename(f)}").read().split()))
    selisih = max(selisih, max(abs(x-y) for x, y in zip(a, b)))
print("selisih terbesar setelah bolak-balik:", selisih, "-> OK" if selisih < 1e-5 else "-> ADA MASALAH")

## 4. Tahap 3–4 — Latih E1, E2, E3
Data, epoch, batch, dan seed **sama** di ketiganya; hanya satu hal yang berbeda.

| Eksperimen | model | beda |
|---|---|---|
| E1_pretrained | `yolo26n.pt` | baseline (transfer learning dari COCO) |
| E2_scratch | `yolo26n.yaml` | bobot acak, tanpa transfer learning |
| E3_tuning | `yolo26n.pt` | **satu** perubahan: `imgsz=640` (baseline 416) |

Kalau `yolo26n.pt` tidak ditemukan, jalankan ulang `pip install -U ultralytics` (versinya harus baru).

In [ ]:
MODEL_PT, MODEL_YAML = "yolo26n.pt", "yolo26n.yaml"
!yolo detect train data={DATA} model={MODEL_PT} epochs=30 imgsz=416 batch=16 seed=0 project=/content/runs name=E1_pretrained exist_ok=True

In [ ]:
!yolo detect train data={DATA} model={MODEL_YAML} pretrained=False epochs=30 imgsz=416 batch=16 seed=0 project=/content/runs name=E2_scratch exist_ok=True

In [ ]:
# E3: pretrained + SATU perubahan saja (imgsz 416 -> 640)
!yolo detect train data={DATA} model={MODEL_PT} epochs=30 imgsz=640 batch=16 seed=0 project=/content/runs name=E3_tuning exist_ok=True

## 5. Tahap 5 — Bandingkan di data VAL dan pilih model terbaik
Model dipilih berdasarkan **mAP50-95 di val**.

In [ ]:
from ultralytics import YOLO
import pandas as pd

eksp = {"E1_pretrained": ("- (baseline)", 416), "E2_scratch": ("bobot acak", 416), "E3_tuning": ("imgsz=640", 640)}
baris = []
for e, (ubah, sz) in eksp.items():
    r = YOLO(f"/content/runs/{e}/weights/best.pt").val(data=DATA, split="val", imgsz=sz, plots=False, verbose=False)
    baris.append(dict(Eksperimen=e, Perubahan=ubah, mAP50=r.box.map50, mAP50_95=r.box.map,
                      P=r.box.mp, R=r.box.mr, ms_per_gambar=r.speed["inference"]))
tabel = pd.DataFrame(baris).round(3)
terbaik = tabel.sort_values("mAP50_95", ascending=False).iloc[0]["Eksperimen"]
display(tabel)
print("Model terpilih (mAP50-95 val tertinggi):", terbaik)

## 6. Uji di data TEST — **sekali saja**
Jangan jalankan sel ini berulang untuk memilih model; kalau diulang, test berubah jadi val kedua dan angkanya tidak jujur.

In [ ]:
sz_terbaik = eksp[terbaik][1]
rt = YOLO(f"/content/runs/{terbaik}/weights/best.pt").val(data=DATA, split="test", imgsz=sz_terbaik,
                                                         project="/content/runs", name="TEST_"+terbaik, exist_ok=True)
print(f"TEST  mAP50={rt.box.map50:.3f}  mAP50-95={rt.box.map:.3f}  P={rt.box.mp:.3f}  R={rt.box.mr:.3f}")
print("per kelas mAP50-95:", dict(zip([names[i] for i in rt.ap_class_index], rt.box.maps[rt.ap_class_index].round(3))))

## 7. Tahap 6 — Simpan hasil (hasil_praktik_<NIM>.txt/.csv)

In [ ]:
import hashlib
sidik = hashlib.md5(b"".join(open(f, "rb").read() for f in sorted(glob.glob(f"{R}/labels/*/*.txt")))).hexdigest()[:10]
tabel.to_csv(f"/content/hasil_praktik_{NIM}.csv", index=False)
with open(f"/content/hasil_praktik_{NIM}.txt", "w") as f:
    f.write(f"NIM / Nama : {NIM} / {NAMA}\nDataset    : dataset_v1 (sidik jari label: {sidik})\n\n")
    f.write(tabel.to_string(index=False))
    f.write(f"\n\nModel terpilih (mAP50-95 val tertinggi): {terbaik}\n")
    f.write(f"Hasil di data TEST (sekali saja)       : mAP50={rt.box.map50:.3f} mAP50-95={rt.box.map:.3f} P={rt.box.mp:.3f} R={rt.box.mr:.3f}\n")
print(open(f"/content/hasil_praktik_{NIM}.txt").read())

## 8. Lihat grafik & contoh prediksi (untuk analisis di README)

In [ ]:
from IPython.display import Image, display
for e in ["E1_pretrained", "E3_tuning"]:
    print(e); display(Image(f"/content/runs/{e}/results.png", width=900))
print("Confusion matrix model terpilih (val):"); display(Image(f"/content/runs/{terbaik}/confusion_matrix.png", width=500))

In [ ]:
# contoh prediksi pada gambar test
m = YOLO(f"/content/runs/{terbaik}/weights/best.pt")
hasil = m.predict(sorted(glob.glob(f"{R}/images/test/*.jpg")), imgsz=sz_terbaik, conf=0.25, save=True,
                  project="/content/runs", name="prediksi_test", exist_ok=True, verbose=False)
plt.figure(figsize=(16, 6))
for i, f in enumerate(sorted(glob.glob("/content/runs/prediksi_test/*.jpg"))[:10]):
    plt.subplot(2, 5, i+1); plt.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.show()

## 9. Kumpulkan & download

In [ ]:
!rm -rf /content/hasil_praktik_{NIM} && mkdir -p /content/hasil_praktik_{NIM}
!cp -r /content/runs/E1_pretrained /content/runs/E2_scratch /content/runs/E3_tuning /content/runs/TEST_{terbaik} /content/hasil_praktik_{NIM}/
!cp /content/hasil_praktik_{NIM}.txt /content/hasil_praktik_{NIM}.csv /content/hasil_praktik_{NIM}/
!cp /content/annotations_coco.json /content/hasil_praktik_{NIM}/
!zip -rq /content/hasil_praktik_{NIM}.zip /content/hasil_praktik_{NIM}
files.download(f"/content/hasil_praktik_{NIM}.zip")

In [ ]:
# (Opsional) push langsung ke GitHub. Butuh Personal Access Token (scope: repo).
from getpass import getpass
token = getpass("GitHub token: ")
!git clone -q https://{token}@github.com/Theaaaa88/Computer-Vision-and-Deep-Learning-RET503.git /content/repo
!mkdir -p /content/repo/yolo && cp -r /content/hasil_praktik_{NIM} /content/repo/yolo/
!cp -r /content/dataset_v1 /content/repo/yolo/
!cd /content/repo && git config user.email "you@example.com" && git config user.name "{NAMA}" && git add yolo && git commit -qm "Praktik terpadu YOLO" && git push -q